# Installing and Importing Libraries


In [0]:
%pip install databricks-feature-engineering
%pip install xgboost


In [0]:
dbutils.library.restartPython()


In [0]:
import mlflow
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
import pandas as pd
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp


# Loading the Model and Prediction Dataset


In this step, the **final model is loaded from MLflow** and the dataset to be used for new predictions is prepared.

The prediction dataset uses the **Feature Store** to retrieve the same features used during training. Records corresponding to the **most recent reference date** are selected, ensuring that prediction is made on the most current data available.

Thus, the process reproduces the structure used in training, but now without the target variable (`label=None`), generating `df_predict`, which will be used by the model to make predictions.


In [0]:
# Change the run ID according to your own
model = mlflow.sklearn.load_model("runs:/d4bbbab833f24b8badb9468b12cba1b9/model")


In [0]:
# Function to import SQL query from a file externo
def import_query(path):
    with open(path) as f:
        return f.read()

# FeatureLookups to retrieve features from different Feature Store tables (renamed columns to English)
feature_lookups = [
    FeatureLookup(table_name="feature_store.credit_score.fs_cadastral", lookup_key=["CLIENT_ID", "DOCUMENT_ID", "REF_DATE"]),
    FeatureLookup(table_name="feature_store.credit_score.fs_temporal", lookup_key=["CLIENT_ID", "DOCUMENT_ID", "REF_DATE"]),
    FeatureLookup(table_name="feature_store.credit_score.fs_income_history", lookup_key=["CLIENT_ID", "DOCUMENT_ID", "REF_DATE"]),
    FeatureLookup(table_name="feature_store.credit_score.fs_income", lookup_key=["CLIENT_ID", "DOCUMENT_ID", "REF_DATE"]),
    FeatureLookup(table_name="feature_store.credit_score.fs_employees", lookup_key=["CLIENT_ID", "DOCUMENT_ID", "REF_DATE"]),
    FeatureLookup(table_name="feature_store.credit_score.fs_payment_history", lookup_key=["CLIENT_ID", "DOCUMENT_ID", "REF_DATE"])
]

fe = FeatureEngineeringClient()

# Query to select the most recent dataset
query = """
    SELECT
        REF_DATE,
        CLIENT_ID,
        DOCUMENT_ID
    FROM feature_store.credit_score.fs_cadastral
    WHERE REF_DATE = (
        SELECT MAX(REF_DATE)
        FROM feature_store.credit_score.fs_cadastral
    )
"""

df = spark.sql(query)

# Create feature set for prediction
predict_set = fe.create_training_set(df=df, feature_lookups=feature_lookups, label=None)
df_predict = predict_set.load_df().toPandas()


In [0]:
df_predict.head()


# Model Prediction


In [0]:
df_predict["pred"] = model.predict(df_predict)
df_predict["proba"] = model.predict_proba(df_predict)[:, 1]


# Final Evaluation


In this step, model predictions are compared with **actual payment outcomes**. For this, prediction data is joined to the payments table through ID_DOCUMENTO.

Actual default is defined considering **5 or more days of delay**. Next, **AUC and KS** metrics are calculated, allowing evaluation of the model's ability to separate non-default and default customers on the most recent dataset.

Thus, a **final validation of model performance on real and recent data** is performed after the entire training process.


In [0]:
pagamentos_df = spark.sql("""
    SELECT ID_DOCUMENTO, DATA_PAGAMENTO, DATA_VENCIMENTO
    FROM credit_score.data.pagamentos
""")

df_predict_spark = spark.createDataFrame(df_predict)
df_predict_joined = df_predict_spark.join(pagamentos_df, df_predict_spark.DOCUMENT_ID == pagamentos_df.ID_DOCUMENTO, how="left")
df_predict = df_predict_joined.toPandas()


In [0]:
df_predict["actual_default"] = ((pd.to_datetime(df_predict["DATA_PAGAMENTO"]) - pd.to_datetime(df_predict["DATA_VENCIMENTO"])).dt.days >= 5).astype(int)


In [0]:
auc = roc_auc_score(df_predict["actual_default"], df_predict["proba"])
ks = ks_2samp(df_predict[df_predict["actual_default"] == 1]["proba"],
              df_predict[df_predict["actual_default"] == 0]["proba"]).statistic
print(f"AUC: {auc:.2f} | KS: {ks:.2f}")
